In [1]:
!pip install entsoe-py

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
from entsoe import EntsoePandasClient
import requests

client = EntsoePandasClient(api_key='KEY')

start = pd.Timestamp('20181001', tz='Europe/Berlin')
end = pd.Timestamp('20260126', tz='Europe/Berlin')
country_code = 'DE_LU'

# takes the sequence 1 prices from entsoe since it matches the data from smard
print("Fetching day ahead prices...")
prices = client.query_day_ahead_prices(country_code, start=start, end=end)

print("Fetching load forecast...")
load_forecast = client.query_load_forecast(country_code, start=start, end=end)

print("Fetching wind and solar forecast...")
wind_solar_forecast = client.query_wind_and_solar_forecast(country_code, start=start, end=end)

# aggregation
df = pd.DataFrame()
df['Price_DayAhead'] = prices
df['Load_Forecast'] = load_forecast

wind_solar_forecast.columns = wind_solar_forecast.columns.str.replace(' ', '_')

df = df.join(wind_solar_forecast)

print(df.head())
print(f"\nDaten von {df.index.min()} bis {df.index.max()}")
print(f"Anzahl Zeilen: {len(df)}")
df.index.name = "Date"
df.to_csv('entsoe_forecasts.csv')

Fetching day ahead prices...


C:\Users\menko\AppData\Roaming\Python\Python313\site-packages\bs4\element.py:1813: ResourceWarning: unclosed <ssl.SSLSocket fd=1196, family=2, type=1, proto=0, laddr=('192.168.178.21', 57269), raddr=('20.23.37.29', 443)>
  def __init__(


Fetching load forecast...
Fetching wind and solar forecast...
                           Price_DayAhead  Load_Forecast  Solar  \
2018-10-01 00:00:00+02:00           59.53            NaN    0.0   
2018-10-01 01:00:00+02:00           56.10            NaN    0.0   
2018-10-01 02:00:00+02:00           51.41       42695.57    0.0   
2018-10-01 03:00:00+02:00           47.38       42746.84    0.0   
2018-10-01 04:00:00+02:00           47.59       44038.05    0.0   

                           Wind_Offshore  Wind_Onshore  
2018-10-01 00:00:00+02:00        1635.76       4287.76  
2018-10-01 01:00:00+02:00        1602.39       4148.19  
2018-10-01 02:00:00+02:00        1736.57       4004.51  
2018-10-01 03:00:00+02:00        1792.04       4333.29  
2018-10-01 04:00:00+02:00        2059.44       4611.93  

Daten von 2018-10-01 00:00:00+02:00 bis 2026-01-25 23:45:00+01:00
Anzahl Zeilen: 72597


In [3]:
import pandas as pd
import requests
from datetime import date, timedelta

cities = {
    "Berlin": {"lat": 52.52, "lon": 13.41},
    "Hamburg": {"lat": 53.55, "lon": 9.99},
    "Köln": {"lat": 50.93, "lon": 6.95},
    "München": {"lat": 48.13, "lon": 11.58},
    "Leipzig": {"lat": 51.33, "lon": 12.37},
    "Frankfurt": {"lat": 50.11, "lon": 8.68}
}

yesterday = (date.today() - timedelta(days=1)).isoformat()
print(f"Hole historische Wetterdaten bis: {yesterday}")
print(f"Hole Wettervorhersagen für die nächsten 2 Tage (heute + morgen)")

def get_weather_forecasts():
    all_city_data = []
    
    for name, coord in cities.items():
        # Historische Forecast-Daten bis gestern
        url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coord["lat"],
            "longitude": coord["lon"],
            "start_date": "2018-10-01",
            "end_date": yesterday,
            "hourly": "temperature_2m,windspeed_100m,shortwave_radiation",
            "timezone": "Europe/Berlin"
        }        
        response = requests.get(url, params=params).json()
        hourly = response.get('hourly', {})
        
        df_city = pd.DataFrame({
            "timestamp": pd.to_datetime(hourly.get('time')),
            f"temp_{name}": hourly.get('temperature_2m'),
            f"wind_{name}": hourly.get('windspeed_100m'),
            f"solar_{name}": hourly.get('shortwave_radiation')
        })
        df_city.set_index("timestamp", inplace=True)
        all_city_data.append(df_city)
        
        # Aktuelle Wettervorhersage für heute und morgen (forecast_days=2)
        url_forecast = "https://api.open-meteo.com/v1/forecast"
        params_forecast = {
            "latitude": coord["lat"],
            "longitude": coord["lon"],
            "hourly": "temperature_2m,windspeed_100m,shortwave_radiation",
            "forecast_days": 2,  # Heute (23.) + Morgen (24.)
            "timezone": "Europe/Berlin"
        }
        response_forecast = requests.get(url_forecast, params=params_forecast).json()
        hourly_forecast = response_forecast.get('hourly', {})
        
        df_city_forecast = pd.DataFrame({
            "timestamp": pd.to_datetime(hourly_forecast.get('time')),
            f"temp_{name}": hourly_forecast.get('temperature_2m'),
            f"wind_{name}": hourly_forecast.get('windspeed_100m'),
            f"solar_{name}": hourly_forecast.get('shortwave_radiation')
        })
        df_city_forecast.set_index("timestamp", inplace=True)
        all_city_data.append(df_city_forecast)

    df_combined = pd.concat(all_city_data, axis=1)
    df_combined = df_combined.T.groupby(level=0).first().T
    
    df_final = pd.DataFrame(index=df_combined.index)
    df_final['Weather_Temp_Forecast'] = df_combined[[col for col in df_combined.columns if 'temp' in col]].mean(axis=1)
    df_final['Weather_Wind_Forecast'] = df_combined[[col for col in df_combined.columns if 'wind' in col]].mean(axis=1)
    df_final['Weather_Solar_Forecast'] = df_combined[[col for col in df_combined.columns if 'solar' in col]].mean(axis=1)
    
    return df_final

weather_df = get_weather_forecasts()
weather_df = weather_df.round(1)
print(f"\nWetterdaten von {weather_df.index.min()} bis {weather_df.index.max()}")
print(f"Anzahl Zeilen: {len(weather_df)}")
weather_df.to_csv("weather_forecasts_aggregated.csv")

Hole historische Wetterdaten bis: 2026-01-23
Hole Wettervorhersagen für die nächsten 2 Tage (heute + morgen)

Wetterdaten von 2018-10-01 00:00:00 bis 2026-01-25 23:00:00
Anzahl Zeilen: 64176
